In [1]:
!pip install -q -U transformers datasets accelerate peft sentence-transformers wandb 


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, gc, re, random, warnings
import numpy as np, pandas as pd, torch

In [ ]:
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    print("W&B logged in via secret")
except Exception as e:
    print("Secret not found, will prompt interactively:", e)

In [ ]:

warnings.filterwarnings("ignore")

class CFG:
    # --- data ---
    DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"  # change if your dataset folder name differs
    OPTS     = ["A", "B", "C", "D", "E"]                   # the five answer choices
    SEED     = 42                                          # fixes randomness -> reproducible

    # --- score-driver model (DeBERTa + LoRA, Milestone 4) ---
    MODEL_NAME = "microsoft/deberta-v3-base"
    MAX_LEN    = 256      # max tokens per (question + one option) pair
    EPOCHS     = 3        # passes over the training data
    LR         = 2e-4     # LoRA uses a higher learning rate than full fine-tuning
    TRAIN_BS   = 4        # training batch size
    EVAL_BS    = 8        # evaluation batch size

    # --- pretrained zero-shot model (Milestone 2) ---
    ST_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"

    # --- Weights & Biases ---
    WANDB_PROJECT = "23f2002523-t22026"   # <-- put YOUR W&B project name here

def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_everything(CFG.SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("torch:", torch.__version__)

In [ ]:
train = pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
samp  = pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("train shape:", train.shape)   # (rows, columns)
print("test shape :", test.shape)
print("columns    :", list(train.columns))

# how often each letter is the correct answer (checks for imbalance)
print("\nAnswer counts:")
print(train["answer"].value_counts().sort_index())

# check for any missing/empty cells
print("\nMissing values -> train:", train.isnull().sum().sum(),
      "| test:", test.isnull().sum().sum())

# peek at one full example
print("\n--- Example row 0 ---")
print("PROMPT:", train.loc[0, "prompt"][:200])
for o in CFG.OPTS:
    print(f"{o}:", str(train.loc[0, o])[:120])
print("ANSWER:", train.loc[0, "answer"])

train.head(2)